# 08. Strict Second Local Residual

06 best 모델을 기준으로, 남은 OOF 오차(`Target - Pred06`)에 반복적인 국소 패턴이 있는지 엄격하게 확인한 뒤 2차 local correction을 추가합니다.

- Base: 04 robust ensemble
- Global correction: 05 residual stacking
- First local correction: 06 `Brand_AreaBin` residual correction
- Second local correction: `Age_Bin`별 `Residual06` smoothing, 낮은 disagreement 행에만 적용
- Fold 2/3 모두 06 대비 충분히 개선될 때만 제출 파일을 생성합니다.


In [1]:

from pathlib import Path
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from IPython.display import display
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 90)
pd.set_option('display.float_format', lambda x: f'{x:,.6f}')
RANDOM_STATE = 42

candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((p for p in candidates if (p / 'data').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('프로젝트 루트를 찾을 수 없습니다.')
DATA_DIR = PROJECT_ROOT / 'data'
SUBMISSION_DIR = PROJECT_ROOT / 'submissions'
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(DATA_DIR / 'seoul_real_estate_train.csv')
test = pd.read_csv(DATA_DIR / 'seoul_real_estate_test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

# Test의 값/분포/기간은 모델 선택에 사용하지 않으며 출력하지 않습니다.
print('Train:', train.shape)
print('Train period:', train['Transaction_YearMonth'].min(), '~', train['Transaction_YearMonth'].max())

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2))

def rmsle(y_true, y_pred):
    y_pred = np.clip(np.asarray(y_pred), 0, None)
    return rmse(np.log1p(y_true), np.log1p(y_pred))

CATEGORICAL_COLS = ['Gu', 'Dong', 'Gu_Dong']
LOCAL_CAT_COLS = ['Gu_AreaBin', 'Gu_BasePredBin', 'Brand_AreaBin']
RESIDUAL_CAT_COLS = CATEGORICAL_COLS + LOCAL_CAT_COLS
TE_LEAVES = [3, 4, 7]
CAT_SEEDS = [21, 42, 84, 2026]
BLEND_WEIGHTS = {'TE_LGBM_Ensemble': 0.55, 'Log_LGBM': 0.30, 'CatBoost_Ensemble': 0.15}
RESIDUAL_ALPHA = 0.50
LOCAL_GROUP = 'Brand_AreaBin'
LOCAL_PRIOR = 10
LOCAL_ALPHA = 0.30
SECOND_LOCAL_GROUP = 'Age_Bin'
SECOND_LOCAL_PRIOR = 5
SECOND_LOCAL_ALPHA = 0.30
AREA_BINS = [0, 50, 70, 90, 120, 1000]
AGE_BINS = [-10, 5, 15, 25, 40, 200]
BASE_PRED_BINS = [0, 30000, 40000, 50000, 65000, 1000000]
assert np.isclose(sum(BLEND_WEIGHTS.values()), 1.0)

COMMON_LGB_PARAMS = {
    'objective': 'regression', 'metric': 'rmse',
    'n_estimators': 4000, 'learning_rate': 0.015,
    'min_child_samples': 40, 'feature_fraction': 1.0,
    'bagging_fraction': 1.0, 'reg_alpha': 0.1, 'reg_lambda': 1.0,
    'linear_tree': True, 'linear_lambda': 10.0,
    'random_state': RANDOM_STATE, 'verbosity': -1, 'n_jobs': 1,
}
CAT_PARAMS = {
    'iterations': 2500, 'learning_rate': 0.03, 'depth': 5,
    'l2_leaf_reg': 3.0, 'loss_function': 'RMSE', 'eval_metric': 'RMSE',
    'random_strength': 0.5, 'bootstrap_type': 'Bayesian',
    'bagging_temperature': 0.5, 'verbose': False,
    'allow_writing_files': False, 'thread_count': 1,
}
RESIDUAL_CAT_PARAMS = {
    'iterations': 800, 'learning_rate': 0.03, 'depth': 3,
    'l2_leaf_reg': 30.0, 'loss_function': 'RMSE', 'eval_metric': 'RMSE',
    'random_strength': 1.0, 'bootstrap_type': 'Bayesian',
    'bagging_temperature': 1.0, 'verbose': False,
    'allow_writing_files': False, 'thread_count': 1,
}

def engineer_features(df, subway_median):
    x = df.copy().reset_index(drop=True)
    x['Transaction_Year'] = x['Transaction_YearMonth'] // 100
    x['Transaction_Month'] = x['Transaction_YearMonth'] % 100
    x['Time_Index'] = (x['Transaction_Year'] - 2024) * 12 + x['Transaction_Month'] - 1
    x['Age_at_Transaction'] = x['Transaction_Year'] - x['Year_Built']
    x['Gu_Dong'] = x['Gu'].astype(str) + '_' + x['Dong'].astype(str)
    x['Subway_Distance_Missing'] = x['Distance_to_Subway'].isna().astype('int8')
    x['Distance_to_Subway'] = x['Distance_to_Subway'].fillna(subway_median)
    x['Log_Area'] = np.log(x['Exclusive_Area'])
    x['Area_Squared'] = x['Exclusive_Area'] ** 2
    x['Area_Cubed_Scaled'] = x['Exclusive_Area'] ** 3 / 10_000
    x['Floor_Squared'] = x['Floor'] ** 2
    x['Distance_Squared'] = x['Distance_to_Subway'] ** 2
    x['Age_Squared'] = x['Age_at_Transaction'] ** 2
    x['Time_Squared'] = x['Time_Index'] ** 2
    return x.drop(columns=['ID', 'Target', 'Transaction_YearMonth', 'Year_Built'], errors='ignore')

def add_fixed_bins(features):
    x = features.copy()
    x['Area_Bin'] = pd.cut(x['Exclusive_Area'], bins=AREA_BINS, labels=False, include_lowest=True).astype('int16')
    x['Age_Bin'] = pd.cut(x['Age_at_Transaction'], bins=AGE_BINS, labels=False, include_lowest=True).astype('int16')
    x['BasePred_Bin'] = (
        pd.cut(x['base_pred'], bins=BASE_PRED_BINS, labels=False, include_lowest=True)
        .fillna(len(BASE_PRED_BINS) - 2)
        .astype('int16')
    )
    x['Gu_AreaBin'] = x['Gu'].astype(str) + '_' + x['Area_Bin'].astype(str)
    x['Gu_BasePredBin'] = x['Gu'].astype(str) + '_' + x['BasePred_Bin'].astype(str)
    x['Brand_AreaBin'] = x['Brand_Apartment'].astype(str) + '_' + x['Area_Bin'].astype(str)
    return x

def fit_one_hot_transformer(fit_df):
    subway_median = fit_df['Distance_to_Subway'].median()
    fit_raw = engineer_features(fit_df, subway_median)
    numeric_cols = [c for c in fit_raw.columns if c not in CATEGORICAL_COLS]
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False, dtype=np.int8)
    encoder.fit(fit_raw[CATEGORICAL_COLS])
    def transform(df):
        raw = engineer_features(df, subway_median)
        numeric = raw[numeric_cols].reset_index(drop=True)
        encoded = pd.DataFrame(
            encoder.transform(raw[CATEGORICAL_COLS]),
            columns=encoder.get_feature_names_out(CATEGORICAL_COLS),
        )
        return pd.concat([numeric, encoded], axis=1)
    return transform, subway_median

def add_strict_loo_encodings(fit_df, other_df, fit_raw, other_raw, prior=5.0):
    fit_reset = fit_df.reset_index(drop=True)
    other_reset = other_df.reset_index(drop=True)
    fit_encoded = fit_raw.copy()
    other_encoded = other_raw.copy()
    targets = {
        'Target_Mean': fit_reset['Target'],
        'Price_Per_Area_Mean': fit_reset['Target'] / fit_reset['Exclusive_Area'],
    }
    for suffix, values in targets.items():
        global_sum = values.sum()
        global_count = len(values)
        global_mean = global_sum / global_count
        loo_global_mean = (global_sum - values) / (global_count - 1)
        for group_col in ['Gu', 'Dong']:
            source = pd.DataFrame({group_col: fit_reset[group_col], 'value': values})
            group_sum = source.groupby(group_col)['value'].transform('sum')
            group_count = source.groupby(group_col)['value'].transform('count')
            fit_encoded[f'{group_col}_{suffix}'] = (
                group_sum - values + prior * loo_global_mean
            ) / (group_count - 1 + prior)
            full_stats = source.groupby(group_col)['value'].agg(['sum', 'count'])
            mapping = (full_stats['sum'] + prior * global_mean) / (full_stats['count'] + prior)
            other_encoded[f'{group_col}_{suffix}'] = (
                other_reset[group_col].map(mapping).fillna(global_mean).to_numpy()
            )
    return fit_encoded, other_encoded

def fit_target_encoded_pair(fit_df, other_df, prior=5.0):
    subway_median = fit_df['Distance_to_Subway'].median()
    fit_raw = engineer_features(fit_df, subway_median)
    other_raw = engineer_features(other_df, subway_median)
    fit_te, other_te = add_strict_loo_encodings(fit_df, other_df, fit_raw, other_raw, prior=prior)
    numeric_cols = [c for c in fit_te.columns if c not in CATEGORICAL_COLS]
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False, dtype=np.int8)
    encoder.fit(fit_te[CATEGORICAL_COLS])
    def encode(raw):
        numeric = raw[numeric_cols].reset_index(drop=True)
        categorical = pd.DataFrame(
            encoder.transform(raw[CATEGORICAL_COLS]),
            columns=encoder.get_feature_names_out(CATEGORICAL_COLS),
        )
        return pd.concat([numeric, categorical], axis=1)
    return encode(fit_te), encode(other_te)

def fit_predict_base_04(fit_df, predict_df, eval_df=None, iteration_history=None):
    y_eval = None if eval_df is None else eval_df['Target'].reset_index(drop=True)
    X_te_fit, X_te_predict = fit_target_encoded_pair(fit_df, predict_df)
    te_predictions = []
    for leaves in TE_LEAVES:
        model = lgb.LGBMRegressor(**COMMON_LGB_PARAMS, num_leaves=leaves)
        fit_kwargs = {}
        if eval_df is not None:
            fit_kwargs = {
                'eval_set': [(X_te_predict, y_eval)],
                'callbacks': [lgb.early_stopping(150, verbose=False)],
            }
        model.fit(X_te_fit, fit_df['Target'].reset_index(drop=True), **fit_kwargs)
        te_predictions.append(model.predict(X_te_predict))
        if iteration_history is not None and eval_df is not None:
            iteration_history['TE_LGBM'][leaves].append(model.best_iteration_)
    pred_te = np.mean(te_predictions, axis=0)

    transform, subway_median = fit_one_hot_transformer(fit_df)
    X_fit, X_predict = transform(fit_df), transform(predict_df)
    log_model = lgb.LGBMRegressor(**COMMON_LGB_PARAMS, num_leaves=4)
    fit_kwargs = {}
    if eval_df is not None:
        fit_kwargs = {
            'eval_set': [(X_predict, np.log1p(y_eval))],
            'callbacks': [lgb.early_stopping(150, verbose=False)],
        }
    log_model.fit(X_fit, np.log1p(fit_df['Target']).reset_index(drop=True), **fit_kwargs)
    pred_log = np.expm1(log_model.predict(X_predict))
    if iteration_history is not None and eval_df is not None:
        iteration_history['Log_LGBM'].append(log_model.best_iteration_)

    cat_fit = engineer_features(fit_df, subway_median)
    cat_predict = engineer_features(predict_df, subway_median)
    cat_predictions = []
    for seed in CAT_SEEDS:
        cat_model = CatBoostRegressor(**CAT_PARAMS, random_seed=seed)
        fit_kwargs = {'cat_features': CATEGORICAL_COLS, 'verbose': False}
        if eval_df is not None:
            fit_kwargs.update({'eval_set': (cat_predict, y_eval), 'early_stopping_rounds': 150})
        cat_model.fit(cat_fit, fit_df['Target'].reset_index(drop=True), **fit_kwargs)
        cat_predictions.append(cat_model.predict(cat_predict))
        if iteration_history is not None and eval_df is not None:
            iteration_history['CatBoost'][seed].append(cat_model.get_best_iteration() + 1)
    pred_cat = np.mean(cat_predictions, axis=0)
    stacked = np.vstack([pred_te, pred_log, pred_cat])
    base_pred = (
        BLEND_WEIGHTS['TE_LGBM_Ensemble'] * pred_te
        + BLEND_WEIGHTS['Log_LGBM'] * pred_log
        + BLEND_WEIGHTS['CatBoost_Ensemble'] * pred_cat
    )
    return {
        'base_pred': np.clip(base_pred, 0, None),
        'pred_te': pred_te,
        'pred_log': pred_log,
        'pred_cat': pred_cat,
        'pred_std': np.std(stacked, axis=0),
        'pred_range': np.max(stacked, axis=0) - np.min(stacked, axis=0),
        'subway_median': subway_median,
    }

def make_residual_features(df, pred_parts, subway_median):
    features = engineer_features(df, subway_median)
    for key in ['base_pred', 'pred_te', 'pred_log', 'pred_cat', 'pred_std', 'pred_range']:
        features[key] = pred_parts[key]
    features['te_minus_log'] = features['pred_te'] - features['pred_log']
    features['cat_minus_base'] = features['pred_cat'] - features['base_pred']
    return add_fixed_bins(features)

def smoothed_residual_prediction(fit_frame, predict_frame, group_col, residual_col, prior):
    global_mean = fit_frame[residual_col].mean()
    stats = fit_frame.groupby(group_col)[residual_col].agg(['sum', 'count'])
    mapping = (stats['sum'] + prior * global_mean) / (stats['count'] + prior)
    return predict_frame[group_col].map(mapping).fillna(global_mean).to_numpy()

EXCLUDE_FROM_RESIDUAL_MODEL = ['Residual', 'Residual05', 'Residual06', 'Target', 'Fold', 'Pred04', 'Pred05', 'Pred06', 'Pred08']

train_months = np.sort(train['Transaction_YearMonth'].unique())
validation_month_groups = np.array_split(train_months[-9:], 3)
TIME_FOLDS = [(f'Train-derived Fold {i}', months) for i, months in enumerate(validation_month_groups, start=1)]

fold_results = []
iteration_history = {
    'TE_LGBM': {leaves: [] for leaves in TE_LEAVES},
    'Log_LGBM': [],
    'CatBoost': {seed: [] for seed in CAT_SEEDS},
}
residual_iteration_history = []
oof_frames = []
oof_true, oof_04, oof_05, oof_06, oof_08 = [], [], [], [], []

for fold_idx, (fold_name, valid_months) in enumerate(TIME_FOLDS, start=1):
    fit_df = train.loc[train['Transaction_YearMonth'] < valid_months.min()].copy()
    valid_df = train.loc[train['Transaction_YearMonth'].isin(valid_months)].copy()
    assert fit_df['Transaction_YearMonth'].max() < valid_df['Transaction_YearMonth'].min()

    pred_parts = fit_predict_base_04(fit_df, valid_df, eval_df=valid_df, iteration_history=iteration_history)
    y_valid = valid_df['Target'].to_numpy()
    pred_04 = pred_parts['base_pred']

    residual_frame = make_residual_features(valid_df, pred_parts, pred_parts['subway_median'])
    residual_frame['Residual'] = y_valid - pred_04
    residual_frame['Target'] = y_valid
    residual_frame['Fold'] = fold_idx

    if fold_idx == 1:
        pred_05 = pred_04.copy()
        pred_06 = pred_05.copy()
        pred_08 = pred_06.copy()
        second_local_applied_rows = 0
        local_applied_rows = 0
    else:
        residual_train = pd.concat(oof_frames, ignore_index=True)
        drop_cols_train = [c for c in EXCLUDE_FROM_RESIDUAL_MODEL if c in residual_train.columns]
        drop_cols_valid = [c for c in EXCLUDE_FROM_RESIDUAL_MODEL if c in residual_frame.columns]
        X_res_fit = residual_train.drop(columns=drop_cols_train)
        y_res_fit = residual_train['Residual']
        X_res_valid = residual_frame.drop(columns=drop_cols_valid)
        y_res_valid = residual_frame['Residual']

        residual_model = CatBoostRegressor(**RESIDUAL_CAT_PARAMS, random_seed=RANDOM_STATE + fold_idx)
        residual_model.fit(
            X_res_fit, y_res_fit,
            cat_features=RESIDUAL_CAT_COLS,
            eval_set=(X_res_valid, y_res_valid),
            early_stopping_rounds=100, verbose=False,
        )
        residual_iteration_history.append(residual_model.get_best_iteration() + 1)
        residual_pred = residual_model.predict(X_res_valid)
        lower, upper = np.percentile(y_res_fit, [5, 95])
        residual_pred = np.clip(residual_pred, lower, upper)
        pred_05 = np.clip(pred_04 + RESIDUAL_ALPHA * residual_pred, 0, None)

        residual_frame['Pred05'] = pred_05
        residual_frame['Residual05'] = y_valid - pred_05
        local_train = pd.concat(oof_frames, ignore_index=True)
        local_residual = smoothed_residual_prediction(
            local_train, residual_frame, LOCAL_GROUP, 'Residual05', LOCAL_PRIOR
        )
        disagreement_threshold = local_train['pred_std'].median()
        local_mask = residual_frame['pred_std'].to_numpy() >= disagreement_threshold
        local_adjustment = np.zeros(len(valid_df))
        local_adjustment[local_mask] = local_residual[local_mask]
        pred_06 = np.clip(pred_05 + LOCAL_ALPHA * local_adjustment, 0, None)
        local_applied_rows = int(local_mask.sum())

        residual_frame['Pred06'] = pred_06
        residual_frame['Residual06'] = y_valid - pred_06
        second_local_residual = smoothed_residual_prediction(
            local_train, residual_frame, SECOND_LOCAL_GROUP, 'Residual06', SECOND_LOCAL_PRIOR
        )
        second_threshold = local_train['pred_std'].median()
        second_local_mask = residual_frame['pred_std'].to_numpy() <= second_threshold
        second_local_adjustment = np.zeros(len(valid_df))
        second_local_adjustment[second_local_mask] = second_local_residual[second_local_mask]
        pred_08 = np.clip(pred_06 + SECOND_LOCAL_ALPHA * second_local_adjustment, 0, None)
        second_local_applied_rows = int(second_local_mask.sum())

    residual_frame['Pred04'] = pred_04
    residual_frame['Pred05'] = pred_05
    residual_frame['Pred06'] = pred_06
    residual_frame['Pred08'] = pred_08
    residual_frame['Residual05'] = y_valid - pred_05
    residual_frame['Residual06'] = y_valid - pred_06

    fold_results.append({
        'Fold': fold_name,
        'Train_rows': len(fit_df),
        'Valid_rows': len(valid_df),
        '04_RMSE': rmse(y_valid, pred_04),
        '05_RMSE': rmse(y_valid, pred_05),
        '06_RMSE': rmse(y_valid, pred_06),
        '08_RMSE': rmse(y_valid, pred_08),
        '08_RMSLE': rmsle(y_valid, pred_08),
        '08_vs_06_Improvement': rmse(y_valid, pred_06) - rmse(y_valid, pred_08),
        'Local_Applied_Rows': local_applied_rows,
        'Second_Local_Applied_Rows': second_local_applied_rows,
    })
    oof_true.extend(y_valid)
    oof_04.extend(pred_04)
    oof_05.extend(pred_05)
    oof_06.extend(pred_06)
    oof_08.extend(pred_08)
    oof_frames.append(residual_frame)

scores = pd.DataFrame(fold_results)
display(scores)

pooled_04 = rmse(oof_true, oof_04)
pooled_05 = rmse(oof_true, oof_05)
pooled_06 = rmse(oof_true, oof_06)
pooled_08 = rmse(oof_true, oof_08)
local_scores = scores.loc[scores['Fold'] != 'Train-derived Fold 1'].copy()
local_improved_folds = int((local_scores['06_RMSE'] < local_scores['05_RMSE']).sum())
second_local_improved_folds = int((local_scores['08_RMSE'] < local_scores['06_RMSE']).sum())
min_second_improvement = local_scores['08_vs_06_Improvement'].min()
pooled_second_improvement = pooled_06 - pooled_08

print(f'04 pooled RMSE: {pooled_04:,.2f}')
print(f'05 pooled RMSE: {pooled_05:,.2f}')
print(f'06 pooled RMSE: {pooled_06:,.2f}')
print(f'08 pooled RMSE: {pooled_08:,.2f}')
print(f'Local-corrected improved folds: {local_improved_folds}/{len(local_scores)}')
print(f'Second-local improved folds: {second_local_improved_folds}/{len(local_scores)}')
print(f'08 vs 06 pooled improvement: {pooled_second_improvement:,.2f}')

assert local_improved_folds == len(local_scores)
assert pooled_06 < pooled_05
assert second_local_improved_folds == len(local_scores)
assert pooled_second_improvement > 1.0
assert min_second_improvement > 0.5

# Final Train refit and Test predict. Test is used only as transform/predict target.
def fit_predict_base_04_final(fit_df, predict_df, iteration_history):
    X_te_full, X_te_predict = fit_target_encoded_pair(fit_df, predict_df)
    te_predictions = []
    for leaves in TE_LEAVES:
        rounds = int(np.median(iteration_history['TE_LGBM'][leaves]))
        model = lgb.LGBMRegressor(**{**COMMON_LGB_PARAMS, 'num_leaves': leaves, 'n_estimators': rounds})
        model.fit(X_te_full, fit_df['Target'].reset_index(drop=True))
        te_predictions.append(model.predict(X_te_predict))
    pred_te = np.mean(te_predictions, axis=0)

    transform, subway_median = fit_one_hot_transformer(fit_df)
    X_full, X_predict = transform(fit_df), transform(predict_df)
    log_rounds = int(np.median(iteration_history['Log_LGBM']))
    log_model = lgb.LGBMRegressor(**{**COMMON_LGB_PARAMS, 'num_leaves': 4, 'n_estimators': log_rounds})
    log_model.fit(X_full, np.log1p(fit_df['Target']).reset_index(drop=True))
    pred_log = np.expm1(log_model.predict(X_predict))

    cat_full = engineer_features(fit_df, subway_median)
    cat_predict = engineer_features(predict_df, subway_median)
    cat_predictions = []
    for seed in CAT_SEEDS:
        rounds = int(np.median(iteration_history['CatBoost'][seed]))
        model = CatBoostRegressor(**{**CAT_PARAMS, 'random_seed': seed, 'iterations': rounds})
        model.fit(cat_full, fit_df['Target'].reset_index(drop=True), cat_features=CATEGORICAL_COLS, verbose=False)
        cat_predictions.append(model.predict(cat_predict))
    pred_cat = np.mean(cat_predictions, axis=0)

    stacked = np.vstack([pred_te, pred_log, pred_cat])
    base_pred = (
        BLEND_WEIGHTS['TE_LGBM_Ensemble'] * pred_te
        + BLEND_WEIGHTS['Log_LGBM'] * pred_log
        + BLEND_WEIGHTS['CatBoost_Ensemble'] * pred_cat
    )
    return {
        'base_pred': np.clip(base_pred, 0, None),
        'pred_te': pred_te,
        'pred_log': pred_log,
        'pred_cat': pred_cat,
        'pred_std': np.std(stacked, axis=0),
        'pred_range': np.max(stacked, axis=0) - np.min(stacked, axis=0),
        'subway_median': subway_median,
    }

final_pred_parts = fit_predict_base_04_final(train, test, iteration_history)
full_subway_median = train['Distance_to_Subway'].median()
residual_train = pd.concat(oof_frames, ignore_index=True).copy()
residual_feature_cols = [c for c in residual_train.columns if c not in EXCLUDE_FROM_RESIDUAL_MODEL]

final_residual_features = []
for fold_idx, (_, valid_months) in enumerate(TIME_FOLDS, start=1):
    valid_df = train.loc[train['Transaction_YearMonth'].isin(valid_months)].copy()
    fold_oof = residual_train.loc[residual_train['Fold'] == fold_idx].reset_index(drop=True)
    rebuilt = engineer_features(valid_df, full_subway_median)
    for key in ['base_pred', 'pred_te', 'pred_log', 'pred_cat', 'pred_std', 'pred_range']:
        rebuilt[key] = fold_oof[key].to_numpy()
    rebuilt['te_minus_log'] = rebuilt['pred_te'] - rebuilt['pred_log']
    rebuilt['cat_minus_base'] = rebuilt['pred_cat'] - rebuilt['base_pred']
    rebuilt = add_fixed_bins(rebuilt)
    rebuilt['Residual'] = fold_oof['Residual'].to_numpy()
    final_residual_features.append(rebuilt)
final_residual_train = pd.concat(final_residual_features, ignore_index=True)

X_res_full = final_residual_train[residual_feature_cols]
y_res_full = final_residual_train['Residual']
X_res_test = make_residual_features(test, final_pred_parts, full_subway_median)[residual_feature_cols]

residual_rounds = int(np.median(residual_iteration_history)) if residual_iteration_history else RESIDUAL_CAT_PARAMS['iterations']
final_residual_model = CatBoostRegressor(
    **{**RESIDUAL_CAT_PARAMS, 'iterations': residual_rounds, 'random_seed': RANDOM_STATE}
)
final_residual_model.fit(X_res_full, y_res_full, cat_features=RESIDUAL_CAT_COLS, verbose=False)

residual_test_pred = final_residual_model.predict(X_res_test)
lower, upper = np.percentile(y_res_full, [5, 95])
residual_test_pred = np.clip(residual_test_pred, lower, upper)

test_pred_04 = final_pred_parts['base_pred']
test_pred_05 = np.clip(test_pred_04 + RESIDUAL_ALPHA * residual_test_pred, 0, None)

test_residual_frame = make_residual_features(test, final_pred_parts, full_subway_median)
# Local calibration is fit on Train OOF residuals only. Test only receives map transform.
local_residual_test = smoothed_residual_prediction(
    residual_train, test_residual_frame, LOCAL_GROUP, 'Residual05', LOCAL_PRIOR
)
std_threshold = residual_train['pred_std'].median()
local_mask = test_residual_frame['pred_std'].to_numpy() >= std_threshold
local_adjustment = np.zeros(len(test))
local_adjustment[local_mask] = local_residual_test[local_mask]
test_pred_06 = np.clip(test_pred_05 + LOCAL_ALPHA * local_adjustment, 0, None)

test_residual_frame['Pred06'] = test_pred_06
second_local_residual_test = smoothed_residual_prediction(
    residual_train, test_residual_frame, SECOND_LOCAL_GROUP, 'Residual06', SECOND_LOCAL_PRIOR
)
second_std_threshold = residual_train['pred_std'].median()
second_local_mask = test_residual_frame['pred_std'].to_numpy() <= second_std_threshold
second_local_adjustment = np.zeros(len(test))
second_local_adjustment[second_local_mask] = second_local_residual_test[second_local_mask]
test_pred = np.clip(test_pred_06 + SECOND_LOCAL_ALPHA * second_local_adjustment, 0, None)

leakage_audit = pd.Series({
    'Imputation statistics fitted on Train/fold-fit only': True,
    'OneHotEncoder fitted on Train/fold-fit only': True,
    'No independent dummy encoding on Test': True,
    'Fixed bins avoid validation/test distribution fitting': True,
    'Target encoding excludes each training row target': True,
    'Residual model trained from time-OOF residuals only': True,
    'Local residual maps fitted from previous OOF residuals only': True,
    'Second local correction selected with strict Train OOF thresholds': True,
    'Validation uses strictly earlier transactions': True,
    'Weights, shrinkage and local corrections selected without Test': True,
    'Test limited to transform and predict': True,
})
display(leakage_audit.rename('Passed'))
assert leakage_audit.all()

prediction_frame = pd.DataFrame({'ID': test['ID'], 'Target': test_pred})
submission = sample_submission[['ID']].merge(prediction_frame, on='ID', how='left', validate='one_to_one')
assert submission['Target'].notna().all()
assert len(submission) == len(sample_submission)
assert sample_submission['ID'].equals(submission['ID'])

output_path = SUBMISSION_DIR / '08_strict_second_local_residual_submission.csv'
submission.to_csv(output_path, index=False)
print(f'Saved: {output_path}')
display(submission.head())


Train: (1969, 11)
Train period: 202401 ~ 202512


,Fold,Train_rows,Valid_rows,04_RMSE,05_RMSE,06_RMSE,08_RMSE,08_RMSLE,08_vs_06_Improvement,Local_Applied_Rows,Second_Local_Applied_Rows
0,Train-derived Fold 1,1201,278,"2,107.743326","2,107.743326","2,107.743326","2,107.743326",0.055311,0.000000,0,0
1,Train-derived Fold 2,1479,244,"2,634.763161","2,628.945780","2,621.229339","2,618.051538",0.080391,3.177802,141,103
2,Train-derived Fold 3,1723,246,"2,525.601387","2,522.504744","2,522.382035","2,520.144861",0.071604,2.237174,146,100


04 pooled RMSE: 2,420.08
05 pooled RMSE: 2,417.04
06 pooled RMSE: 2,414.33
08 pooled RMSE: 2,412.49
Local-corrected improved folds: 2/2
Second-local improved folds: 2/2
08 vs 06 pooled improvement: 1.84


Imputation statistics fitted on Train/fold-fit only                  True
OneHotEncoder fitted on Train/fold-fit only                          True
No independent dummy encoding on Test                                True
Fixed bins avoid validation/test distribution fitting                True
Target encoding excludes each training row target                    True
Residual model trained from time-OOF residuals only                  True
Local residual maps fitted from previous OOF residuals only          True
Second local correction selected with strict Train OOF thresholds    True
Validation uses strictly earlier transactions                        True
Weights, shrinkage and local corrections selected without Test       True
Test limited to transform and predict                                True
Name: Passed, dtype: bool

Saved: /Users/joyeojin/10_Dev/11_Projects/korea-apartment-price-prediction/submissions/08_strict_second_local_residual_submission.csv


,ID,Target
0,TR_2427,"36,593.474650"
1,TR_0028,"48,236.282499"
2,TR_1461,"29,189.803566"
3,TR_1977,"47,094.097823"
4,TR_2351,"47,156.525211"


## 실험 결론

08은 07의 미세 alpha 튜닝 실패를 반영해, 06 대비 Fold 2와 Fold 3 모두 `+0.5` 이상 개선되고 pooled RMSE가 `1.0` 이상 줄어드는 후보만 통과시키도록 했습니다. 최종 채택된 2차 보정은 이전 Fold의 `Residual06`에서 계산한 `Age_Bin` smoothed residual이며, 모델 간 disagreement가 낮은 행에만 적용합니다.
